In [ ]:
import os
import numpy as np
import pandas as pd

from gudhi.representations import Silhouette
from scipy.ndimage import gaussian_filter1d

CONFIG = {
    0: {
        "sigma": 0,
        "resolution": 150,
        "normalization": "none",
        "THR_MODE": "p10",
    },
    1: {
        "sigma": 2,
        "resolution": 150,
        "normalization": "l1",
        "THR_MODE": "p10",
    }
}

base = "RIPS"
dim_intervals = [0, 1]

def read_and_save(filedir, tube):
    if tube and tube[0] != ".":
        _, ext = os.path.splitext(os.path.join(filedir, tube))
        tubenamerips = tube.split("_")[-1].split(".")[0]

        if ext != ".pdf" and tubenamerips == "Rips0":
            r0_path = os.path.join(filedir, "_".join(tube.split("_")[:-1]) + "_Rips0.txt")
            r1_path = os.path.join(filedir, "_".join(tube.split("_")[:-1]) + "_Rips1.txt")

            Rips0 = np.array(pd.read_csv(r0_path, sep=" ", header=None))
            if len(Rips0) and np.isinf(Rips0[-1, 1]):
                Rips0 = Rips0[:-1]

            Rips1 = np.array(pd.read_csv(r1_path, sep=" ", header=None))
            if len(Rips1) and np.isnan(Rips1[-1, 1]):
                Rips1[-1, 1] = 0

            return [[Rips0, Rips1], None, None]
    return []

def normalize_curve(curve, mode="l1"):
    if mode == "none":
        return curve
    if mode == "l1":
        s = np.sum(np.abs(curve))
        return curve / (s + 1e-12)
    raise ValueError("Unknown normalization mode.")

def apply_threshold(pairs, thr):
    if pairs is None or len(pairs) == 0:
        return np.empty((0, 2))
    pairs = np.asarray(pairs, float)
    if pairs.ndim != 2 or pairs.shape[1] < 2:
        return np.empty((0, 2))
    pers = pairs[:, 1] - pairs[:, 0]
    sel = (pers >= thr) & np.isfinite(pers)
    return pairs[sel]

def compute_persistence_silhouette(diagrams, dimension, thr_dim, cfg):
    thr_value = thr_dim[dimension]
    pairs = diagrams[dimension]
    pairs = apply_threshold(pairs, thr_value)

    if pairs is None or len(pairs) == 0:
        return np.zeros(cfg["resolution"])

    S = Silhouette(resolution=cfg["resolution"])
    sil = S.fit_transform([pairs])[0]

    sil = normalize_curve(sil, cfg["normalization"])

    if cfg["sigma"] > 0:
        sil = gaussian_filter1d(sil, sigma=cfg["sigma"])

    return sil

for DIMENSION in dim_intervals:

    cfg = CONFIG[DIMENSION]

    thr_dim = compute_global_thresholds(base, [DIMENSION], cfg["THR_MODE"])

    # NON RELAPSE
    direct = os.path.join(base, "NonRelapse")
    listdirNR = sorted(os.listdir(direct))
    SP_NonRelapse = []

    for patient in listdirNR:
        if patient.startswith("."): continue
        listpac = sorted(os.listdir(os.path.join(direct, patient)))
        for filename in listpac:
            data = read_and_save(os.path.join(direct, patient), filename)
            if data:
                SP_NonRelapse.append(
                    compute_persistence_silhouette(data[0], DIMENSION, thr_dim, cfg)
                )
                break

    # RELAPSE
    direct = os.path.join(base, "Relapse")
    listdirR = sorted(os.listdir(direct))
    SP_Relapse = []

    for patient in listdirR:
        if patient.startswith("."): continue
        listpac = sorted(os.listdir(os.path.join(direct, patient)))
        for filename in listpac:
            data = read_and_save(os.path.join(direct, patient), filename)
            if data:
                SP_Relapse.append(
                    compute_persistence_silhouette(data[0], DIMENSION, thr_dim, cfg)
                )
                break

    folder = f"PSilhouette{DIMENSION}"
    subfolder = os.path.join(base, folder)

    os.makedirs(os.path.join(subfolder, "Relapse"), exist_ok=True)
    os.makedirs(os.path.join(subfolder, "NonRelapse"), exist_ok=True)

    for i, curve in enumerate(SP_Relapse):
        np.savetxt(os.path.join(subfolder, "Relapse", f"{listdirR[i]}.csv"), curve)

    for i, curve in enumerate(SP_NonRelapse):
        np.savetxt(os.path.join(subfolder, "NonRelapse", f"{listdirNR[i]}.csv"), curve)